# Data Cleaning

This notebook merges two raw, independently-sourced datasets into a single, clean, analysis-ready table:

- **World Bank — World Development Indicators**: country-level development indicators (GDP, life expectancy, unemployment, inflation, population, FDI, CO2 emissions, poverty) across 25 years.
- **Heritage Foundation — Index of Economic Freedom**: overall economic freedom score plus 12 sub-components (property rights, government integrity, trade freedom, etc.) by country and year.

The goal is not just to combine the two sources, but to "correct" real-world messiness, such as: inconsistent country names/codes, missing years, methodology changes over time, and structural mismatches between sources.

**Parts covered in this notebook:**
1. Imports, loading, and initial inspection
2. Country name/code standardisation
3. Reshaping from wide to long format
4. Merging both datasets
5. Handling missing values and outliers
6. Exporting the final clean dataset to `data/processed/`

---

## Part I — Imports, Loading, and Initial Info

First things first, I need to load both raw datasets and get some basic information about each dataset.

No transformations or cleaning take place in this section, only initial inspection.

In [ ]:
import pandas as pd
import sys

sys.path.append("..")

from src import cleaning


In [ ]:
wb_df = pd.read_csv("../data/raw/world_bank_indicators.csv")
ef_df = pd.read_csv("../data/raw/economic_freedom_data.csv")

### World Bank - Initial Inspection

In [ ]:
wb_df.head()

In [ ]:
wb_df.info()

In [ ]:
wb_df.describe()

In [ ]:
wb_df.describe(include=["object", "str"])

### Economic Freedom Index - Initial Inspection

In [ ]:
ef_df.head()

In [ ]:
ef_df.info()

In [ ]:
ef_df.describe()

In [ ]:
ef_df.describe(include=["object", "str"])

## Part II - Country Name and Code Standardisation

Before merging, both datasets need to represent countries the same way. World Bank and Heritage Foundation are independent sources, so naming conventions often diverge (for example: "Korea, Rep." vs "South Korea", or "Russian Federation" vs "Russia").

This section:
1. Compares the unique country names/codes in each dataset
2. Identifies mismatches (countries present in one source but not matched in the other)
3. Builds a standardised mapping to align both datasets on a common country identifier

In [ ]:
wb_df["Country Name"].nunique()

In [ ]:
ef_df["Country"].nunique()

In [ ]:
only_in_wb, only_in_ef = cleaning.compare_countries(
    wb_df, ef_df, "Country Name", "Country", "World Bank", "Economic Freedom"
)

In [ ]:
wb_df = cleaning.standardise_country_names(wb_df, "Country Name")

wb_df.head(10)

In [ ]:
only_in_wb, only_in_ef = cleaning.compare_countries(
    wb_df, ef_df, "Country Name", "Country", "World Bank", "Economic Freedom"
)

In [ ]:
rows_to_drop = [
    "Data from database: World Development Indicators",
    "Last Updated: 07/01/2026",
    "American Samoa",
    "French Polynesia",
    "Isle of Man",
    "Virgin Islands (U.S.)",
    "Antigua and Barbuda",
    "Andorra",
    "Channel Islands",
    "Greenland",
    "Aruba",
    "Faroe Islands",
    "Monaco",
    "Northern Mariana Islands",
    "Marshall Islands",
    "West Bank and Gaza",
    "Tuvalu",
    "St. Kitts and Nevis",
    "Nauru",
    "Turks and Caicos Islands",
    "San Marino",
    "Bermuda",
    "Grenada",
    "Guam",
    "New Caledonia",
    "British Virgin Islands",
    "Curacao",
    "Palau",
    "St. Martin (French part)",
    "Sint Maarten (Dutch part)",
    "Puerto Rico (US)",
    "Gibraltar",
    "South Sudan",
    "Cayman Islands",
]

In [ ]:
wb_df = cleaning.drop_unmatched_countries(wb_df, "Country Name", rows_to_drop)
wb_df = wb_df.dropna(subset=["Country Name"])

wb_df.head(10)

In [ ]:
ef_df = cleaning.drop_unmatched_countries(ef_df, "Country", ["Taiwan"])

In [ ]:
only_in_wb, only_in_ef = cleaning.compare_countries(
    wb_df, ef_df, "Country Name", "Country", "World Bank", "Economic Freedom"
)

### Dropping unmatched countries

After standardising country names, a small number of entries remained unmatched between the two datasets:

- **Junk rows:** The World Bank dataset contains metadata footer rows such as "Data from database..." and "Last Updated...", plus a stray `NaN` row. Since these are not countries, they were dropped.
- **Territories and dependencies:** Some countries are present in the World Bank dataset but not on the Economic Freedom Index (e.g. Greenland, Bermuda, Monaco, Puerto Rico). The Index only evaluates sovereign countries with sufficient economic data, so these have no possible match.
- **South Sudan:** A recognised sovereign country, is excluded from the Economic Freedom Index due to insufficient data availability.
- **Taiwan:** Present only in the Economic Freedom Index, has no equivalent entry in the World Bank dataset due to its disputed sovereign status.

These rows were removed from both datasets, since they cannot be meaningfully merged. The final dataset therefore covers only countries present in **both** sources.

---

## Part III  Reshaping from Wide to Long Format

The World Bank dataset is currently in wide format: each row represents a country-indicator pair, with one column per year (e.g. `2001 [YR2001]`, `2002 [YR2002]`, ...). This structure is convenient for browsing but impractical for merging and analysis.

This section reshapes `wb_df` into long format — one row per country, per year, with each indicator as its own column. This is the structure required to merge with the Economic Freedom Index dataset and to support time-series analysis in the next notebook.

**Steps covered in this section:**
1. Melt the year columns into a single `Year` column
2. Pivot indicators from rows into columns
3. Clean up resulting column names and data types